### RAG Fusion Pipeline

This notebook demonstrates a complete RAG (Retrieval-Augmented Generation) pipeline built on top of our custom `RAGFusion` class.
The pipeline uses **LLM-generated sub-queries** to improve retrieval quality, then fuses the results using **Reciprocal Rank Fusion (RRF)**.

**Steps covered:**
1. Load the source PDF
2. Split documents into chunks
3. Generate embeddings and store in ChromaDB
4. Create a similarity-search retriever
5. Apply RAG Fusion (sub-query generation + RRF)
6. Augmentation - build context from retrieved documents
7. Generation - produce a grounded answer using an LLM

### Imports & Setup

In [1]:
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_ollama import OllamaEmbeddings, OllamaLLM
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

from rag_fusion import RAGFusion

# Load OPENAI_API_KEY from the .env file
load_dotenv()

True

### Step 1 - Load the PDF

`PyPDFLoader` reads the PDF and returns one `Document` object per page.

In [2]:
loader = PyPDFLoader("notebooklm_rag.pdf")
pages = loader.load()

print(f"Loaded {len(pages)} page(s) from the PDF.")

Loaded 3 page(s) from the PDF.


### Step 2 - Split Documents into Chunks

Large pages are split into smaller, overlapping chunks so that the retriever can surface focused, relevant passages rather than entire pages.

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunk(s).")

Split into 19 chunk(s).


### Step 3 - Embeddings & Vector Store

Each chunk is converted into a dense vector using OpenAI's `text-embedding-3-small` model and stored in a ChromaDB vector store.
This makes semantic similarity search possible at query time.

In [5]:
embedding_model = OllamaEmbeddings(model="qwen3-embedding:4b")
llm = OllamaLLM(model="deepseek-r1:1.5b", temperature=0)
llm_groq = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="notebooklm_rag"
)

print("Vector store created successfully.")

Vector store created successfully.


### Step 4 - Create the Retriever

We configure a similarity-search retriever with `k=3`, meaning it will return the 3 most relevant chunks for any given query.

In [6]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


### Step 5 - RAG Fusion

`RAGFusion.from_llm` wires up the LLM to generate multiple sub-queries from the original query.
Each sub-query is sent to the retriever independently, and the results are merged using **Reciprocal Rank Fusion (RRF)** - documents that rank highly across multiple sub-queries bubble to the top.

In [9]:
# llm = ChatOpenAI(model="gpt-5-mini")

# Build the RAG Fusion pipeline: LLM generates 2 sub-queries, retrieves docs for each, then fuses
rag_fusion = RAGFusion.from_llm(
    llm=llm_groq,
    retriever=retriever,
    num_subqueries=3,
    k=3
)

In [10]:
query = "How does NotebookLM retrieve relevant information from uploaded documents?"

# This generates sub-queries, retrieves docs for each, and returns RRF-ranked results
fused_docs = rag_fusion.invoke(query)

print(f"Retrieved {len(fused_docs)} fused document(s).")
for i, doc in enumerate(fused_docs):
    print(f"\n--- Document {i + 1} ---")
    print(doc.page_content)

Retrieved 3 fused document(s).

--- Document 1 ---
model can reference the specific chunks it used to generate an answer.
3. How NotebookLM Processes Documents

--- Document 2 ---
material provided by the user, making it particularly useful for researchers, students, and knowledge workers
who need to work deeply with specific documents.
The tool was first introduced in 2023 and has since gained significant attention for its ability to synthesize
information from multiple sources simultaneously. Users can upload PDFs, Google Docs, YouTube
transcripts, and other formats, and NotebookLM will treat these as the authoritative knowledge base for
answering questions.

--- Document 3 ---
How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike gen

### Step 6 - Augmentation

The retrieved chunks are concatenated into a single context string.
This context will be injected into the generation prompt to ground the LLM's answer.

In [11]:
# Join all retrieved chunks into one context block
context = "\n\n".join([doc.page_content for doc in fused_docs])

print(context)

model can reference the specific chunks it used to generate an answer.
3. How NotebookLM Processes Documents

material provided by the user, making it particularly useful for researchers, students, and knowledge workers
who need to work deeply with specific documents.
The tool was first introduced in 2023 and has since gained significant attention for its ability to synthesize
information from multiple sources simultaneously. Users can upload PDFs, Google Docs, YouTube
transcripts, and other formats, and NotebookLM will treat these as the authoritative knowledge base for
answering questions.

How NotebookLM is Performing RAG Under the Hood
1. Introduction to NotebookLM
NotebookLM is an AI-powered research and note-taking tool developed by Google. It is designed to help
users understand complex documents, generate insights, and answer questions based on content that users
upload directly. Unlike general-purpose AI assistants, NotebookLM grounds all of its responses in the source


### Step 7 - Generation

The context and original query are passed to the LLM via a structured prompt.
The LLM is instructed to answer **only** from the provided context and to say `"I don't know"` if the answer isn't there.

In [12]:
query

'How does NotebookLM retrieve relevant information from uploaded documents?'

In [ ]:
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Use ONLY the context provided below to answer the question.
Be clear, concise, and accurate in your response.
If the answer is not present in the context, say "I don't know" - do not make up an answer.

Context:
{context}

Question: {question}

Answer:
""")

# Chain: prompt -> LLM
generation_chain = prompt | llm

response = generation_chain.invoke({"context": context, "question": query})

print(response)

AttributeError: 'str' object has no attribute 'content'

In [15]:
print(response)

NotebookLM retrieves relevant information from uploaded documents by treating each uploaded content (PDFs, Google Docs, YouTube transcripts) as an authoritative source. It processes these documents simultaneously, using them as a knowledge base to answer questions based on the content provided.

Answer: NotebookLM retrieves relevant information from uploaded documents by processing multiple document formats as authoritative sources and aggregating their content for question answering.
